# 29 · Security, reliability, caching, and cost controls

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Test access filtering, input limits, bounded retries and cache isolation; budget requests.


## Step 1 — Filter evidence before retrieval

Document access checks belong in the data/retrieval layer. A prompt telling the model to hide secrets is insufficient.


In [ ]:
import hashlib, json
docs=[{'id':'public','tenant':'A','text':'Public course policy'},
      {'id':'private','tenant':'B','text':'Tenant B confidential policy'}]
def authorized_docs(tenant):
    return [d for d in docs if d['tenant']==tenant]
assert [d['id'] for d in authorized_docs('A')]==['public']
print(authorized_docs('A'))


**Check your result:** Tenant A never receives tenant B evidence.


## Step 2 — Scope cache entries

The cache key includes tenant, model revision, corpus revision and question. Real systems also need expiry, encryption and deletion rules.


In [ ]:
def cache_key(tenant,question,model_revision,corpus_revision):
    payload=json.dumps([tenant,question,model_revision,corpus_revision],separators=(',',':'))
    return hashlib.sha256(payload.encode()).hexdigest()
assert cache_key('A','policy','v1','r1')!=cache_key('B','policy','v1','r1')
assert cache_key('A','policy','v1','r1')!=cache_key('A','policy','v1','r2')
print('Tenant and corpus changes isolate cache entries.')


**Check your result:** Cache entries cannot be reused across the tested tenants or corpus versions.


## Step 3 — Bound retries and validate inputs

Retry only transient errors and only idempotent operations. This deterministic fixture avoids actual network calls. In production honor Retry-After and use bounded exponential backoff with jitter.


In [ ]:
class TransientError(Exception): pass
def bounded_call(fn,max_attempts=3):
    for attempt in range(max_attempts):
        try: return fn()
        except TransientError:
            if attempt==max_attempts-1: raise
attempts=[0]
def fixture():
    attempts[0]+=1
    if attempts[0]<3: raise TransientError('simulated throttling')
    return 'ok'
assert bounded_call(fixture)=='ok' and attempts[0]==3
def validate_input(text):
    if not isinstance(text,str) or not 1<=len(text)<=2000: raise ValueError('Input length invalid')
    return text
print(validate_input('Explain Delta Lake'))


**Check your result:** The simulated transient failure succeeds on attempt three.


## Step 4 — Estimate spend using explicit assumptions

Prices vary; these are invented teaching rates, not Databricks prices. Token estimation by characters is approximate. Read actual endpoint usage before billing analysis.


In [ ]:
input_tokens=1200
output_tokens=200
requests=1000
assumed_input_per_million=0.50
assumed_output_per_million=1.50
estimate=requests*(input_tokens*assumed_input_per_million+output_tokens*assumed_output_per_million)/1_000_000
print('Illustrative cost units:',round(estimate,2))
assert estimate>=0


**Check your result:** The estimate uses only the stated illustrative rates.


## Practice

Add tests for cross-tenant cache access, oversized prompts, permission revocation, malformed tool calls and retry exhaustion.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Keep authorization and logging rules outside the model. Log request IDs, timings, token usage and error classes while redacting user text and secrets.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
